In [1]:
from __future__ import annotations

from enum import Enum

from debugpy.common.log import LEVELS

In [ ]:
import errno
import sys

import lmdb
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

# from scripts.regsetup import description

import record_pb2
from meta import Run, Dir, File

In [ ]:
from db import Db
# reload(db)

db_name = 'test-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

mydb = Db.open(path, readonly=readonly, create=create)
env = mydb.env

In [ ]:
env, env.flags(), env.path(), env.info()

In [ ]:
with env.begin(
    write=False,
    # buffers= ?
) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            if key[:2] == b'r:':
                print(key, value)

In [ ]:
reload(db)

In [ ]:
run_id = '0016'
run = Run(
    run_id,
    path='',
    description='run from jupyter for development',
    specification='bla',
    start_time=1.234,
    end_time=1.234,
    duration=1.234,
    num_dirs=8,
    num_files=24,
    extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
    status='initialized')

from db import Db
key = Db.mk_run_key(run_id)
value = Db.mk_run_rec(run)
# value = msg.SerializeToString()

key, value


In [ ]:
mydb

In [ ]:
mydb.put_run(run)

In [ ]:
rune = mydb.get_run('0016')
rune

In [ ]:
rune.extra.items()

In [ ]:
env.close()

In [ ]:
with env.begin(write=True) as txn:
    txn.put(key, value)

txn

In [ ]:
with env.begin(write=False) as txn:
    c = txn.cursor()
    c.first()

In [ ]:
txn
# txn.stat()

In [ ]:
# Hieronder: spelen met ID classes

In [14]:
from typing import Optional

class Id:

    def __init__(self, val: int = 0, sup: Optional["Id"] = None, ext_len_bytes: int = 2):
        self.val = val
        self.sup = sup
        self.ext_len = ext_len_bytes
        self.len = (sup.len + ext_len_bytes) if sup else ext_len_bytes

    def to_bytes(self) -> bytes:
        if self.sup:
            return self.sup.to_bytes() + self.val.to_bytes(length=self.ext_len)
        return self.val.to_bytes(length=2)

    def to_hex(self) -> str:
        return self.to_bytes().hex()


In [15]:
idr = Id(41891)

In [16]:
idr.to_bytes(), idr.to_hex()

(b'\xa3\xa3', 'a3a3')

In [17]:
idr.len

2

In [18]:
idd = Id(41892, idr)

In [19]:
idd.to_bytes(), idd.to_hex()

(b'\xa3\xa3\xa3\xa4', 'a3a3a3a4')

In [20]:
idd.len

4

In [21]:
idf = Id(41893, idd, ext_len_bytes=4)

In [22]:
idf.to_bytes(), idf.to_hex()

(b'\xa3\xa3\xa3\xa4\x00\x00\xa3\xa5', 'a3a3a3a40000a3a5')

In [23]:
idf.len

8

In [4]:
r = range(5)
type(iter(r))

range_iterator

In [17]:
# i = iter(r)
next(i)

StopIteration: 

In [18]:
class Id (object):
    def __init__(self, val: int = 0, num_bytes: int = 2):
        self.val = val
        self.num_bytes = num_bytes
    def to_bytes(self):
        return self.val.to_bytes(length=self.num_bytes)


In [20]:
id = Id(3)
id.to_bytes()


b'\x00\x03'

In [21]:
id.val

3

In [80]:
from enum import Enum

class SubId(Id):
    class Level(Enum):
        ONE = 1
        TWO = 2

    LEVEL_SIZES = {
        Level.ONE: {"num_bytes": 2, "max_value": 256**2 - 1},
        Level.TWO: {"num_bytes": 4, "max_value": 256**4 - 1},
    }

    _last_id_by_level = {
        Level.ONE: 0,
        Level.TWO: 0,
    }

    def __init__(self, sup: Id):
        self.sup = sup
        if isinstance(sup, SubId):
            assert sup.level == SubId.Level.ONE, "No unexpected nesting level"
            self.level = SubId.Level.TWO
        else:  # sup is of class: Id
            self.level = SubId.Level.ONE

        last_id = SubId._last_id_by_level[self.level]
        val = last_id + 1
        assert val <= self.LEVEL_SIZES[self.level]["MAX_value"], ValueError("Exceeds max value")
        super().__init__(val, num_bytes=SubId.LEVEL_SIZES[self.level]["num_bytes"])
        SubId._last_id_by_level[self.level] = val

    def to_bytes(self):
        return self.sup.to_bytes() + super().to_bytes()


In [86]:
did = SubId(id)
did.val, did.level, did.to_bytes()
dir(did)

['LEVEL_SIZES',
 'Level',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_last_id_by_level',
 'level',
 'num_bytes',
 'sup',
 'to_bytes',
 'val']

In [72]:
SubId._last_id_by_level

{<Level.ONE: 1>: 2, <Level.TWO: 2>: 0}

In [77]:
fid = SubId(did)
(fid.val, fid.level, SubId._last_id_by_level)

(3, <Level.TWO: 2>, {<Level.ONE: 1>: 2, <Level.TWO: 2>: 3})

In [78]:
fid.to_bytes()


b'\x00\x03\x00\x02\x00\x00\x00\x03'